# 오디오를 이미지로 — 2D 오디오 특징 + CNN 분류 (Colab)

**연습 기법** (IOAI 2025 GAITE **Synthetic Speech Detector** 와 동일): 오디오 분류를 **컴퓨터 비전 문제로**
푼다. 원본 음성을 **멜 스펙트로그램**(가로=시간, 세로=주파수의 2D 이미지)으로 바꾼 뒤 **CNN**(ResNet 등)으로
분류한다. 합성음성 문제는 *미리 변환된* 멜 스펙트로그램 `.pt` 텐서를 주고, 1채널 입력 CNN 으로 bonafide(0)/
spoof(1) 을 가른다(지표 F1).

**대회**: [Don't Call Me Turkey!](https://www.kaggle.com/c/dont-call-me-turkey) — 10초 오디오 클립에 **칠면조
울음**이 있는지 맞히는 **이진 오디오 분류**. 데이터가 **미리 추출된 2D 오디오 특징**(VGGish 임베딩, 최대 10프레임
×128차원)으로 제공돼 ~11MB 로 *매우 가볍고*, 합성음성 문제의 *미리 변환된 멜 .pt* 와 **정확히 같은 구도**다.
그 2D 특징맵을 **이미지처럼** 1채널 CNN 에 넣어 분류한다.

**핵심 흐름 & 배울 점**:
1. **오디오→이미지** — 멜 스펙트로그램 개념(torchaudio 로 합성 신호에서 직접 계산·시각화).
2. **2D 오디오 특징을 고정 크기**(10×128)로 패딩 → 1채널 이미지화.
3. **CNN 분류** — 비전 CNN 으로 이진 분류(합성음성 문제의 ResNet 1채널 `conv1` 과 동형).
4. **임계값 튜닝** — AUC 는 좋아도 0.5 임계값은 F1 에 불리(불균형). val 에서 F1 최대 임계값을 고른다.
5. 지표: 대회는 **AUC**, 합성음성 문제는 **F1** — 둘 다 함께 확인.

> ⚙️ **GPU 권장**(작은 CNN, T4 ~1분). 데이터가 작아 CPU 도 가능.
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.
> 🔑 **첫 실행 시 대회 Rules 수락 필요**: 403 이 나면 [대회 페이지](https://www.kaggle.com/c/dont-call-me-turkey/rules)
> 에서 **"I Understand and Accept"** 1회 클릭 후 다시 실행하세요(자동화 불가).

## 0. 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "torchaudio", "numpy", "pandas", "scikit-learn", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle 에서 데이터 다운로드 (train.json / test.json)
403 이 나면 대회 Rules 를 한 번 수락해야 합니다(위 안내 참조).

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.competition_download_files("dont-call-me-turkey", path=WORK_DIR, quiet=False)
    for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)
    print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith((".json", ".csv"))])
except Exception as e:
    print("⚠️ 다운로드 실패:", repr(e)[:200])
    print("→ 403 이면 https://www.kaggle.com/c/dont-call-me-turkey/rules 에서 'I Understand and Accept' 1회 후 재실행")

## 2. 개념: 오디오 → 멜 스펙트로그램 (audio-as-image)
원본 파형(1D)을 **멜 스펙트로그램**(2D: 시간×멜주파수)으로 바꾸면 *이미지처럼* CNN 에 넣을 수 있다.
합성음성 문제는 이 변환을 미리 해서 `.pt` 로 줬다. 여기선 합성 신호로 변환을 직접 보여준다(다운로드 불필요).

In [ ]:
import numpy as np, torch, torchaudio, matplotlib.pyplot as plt
sr = 16000; t = torch.linspace(0, 1, sr)
wav = torch.sin(2 * np.pi * (200 + 600 * t) * t)[None]          # 올라가는 chirp 신호
mel = torchaudio.transforms.MelSpectrogram(sr, n_mels=64)(wav)   # (1, n_mels, time)
mel_db = torchaudio.transforms.AmplitudeToDB()(mel)[0]
print("멜 스펙트로그램 shape:", tuple(mel.shape), "→ (1, 멜주파수, 시간) = 1채널 이미지")
plt.figure(figsize=(6, 3)); plt.imshow(mel_db, origin="lower", aspect="auto", cmap="magma")
plt.xlabel("time"); plt.ylabel("mel freq"); plt.title("Mel spectrogram of a chirp"); plt.show()

## 3. 미리 추출된 2D 오디오 특징 로드 (VGGish ≈ 멜 .pt)
각 클립은 최대 10프레임 × 128차원 VGGish 임베딩(0~255 정수)이다. **(10, 128) 로 패딩/자르고 0~1 정규화**해
*1채널 이미지*로 만든다. (`bonafide/spoof` 대신 여기선 `not-turkey(0)/turkey(1)`)

In [ ]:
import json, pandas as pd
T = 10  # 프레임 수 고정

def to_array(emb):
    a = np.asarray(emb, dtype="float32")           # (frames, 128)
    if a.ndim == 1: a = a[None, :]
    a = a[:T]
    if len(a) < T:
        a = np.concatenate([a, np.zeros((T - len(a), 128), "float32")], 0)
    return a / 255.0

train = json.load(open(os.path.join(WORK_DIR, "train.json")))
test  = json.load(open(os.path.join(WORK_DIR, "test.json")))
X = np.stack([to_array(c["audio_embedding"]) for c in train]).astype("float32")
y = np.array([int(c["is_turkey"]) for c in train], dtype="int64")
X_test = np.stack([to_array(c["audio_embedding"]) for c in test]).astype("float32")
test_ids = [c["vid_id"] for c in test]
print("X:", X.shape, "| y balance:", np.bincount(y), "| X_test:", X_test.shape)

## 4. 1채널 CNN 정의 (2D 특징맵을 이미지처럼 분류)
합성음성 문제의 *ResNet `conv1`(1, 64, …)* 처럼 **1채널 입력** CNN. 3블록(32→64→128)+드롭아웃으로 표현력 확보.

In [ ]:
import torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 1))
    def forward(self, x):
        return self.head(self.feat(x))

print("AudioCNN 파라미터:", sum(p.numel() for p in AudioCNN().parameters()))

## 5. 학습 & 평가 — val AUC, 그리고 **임계값 튜닝**으로 F1
AUC 는 *확률 순위*만 보지만 F1 은 **임계값**에 민감하다. 클래스가 불균형이라 0.5 가 최적이 아닐 수 있어,
val 에서 **F1 이 최대가 되는 임계값**을 고른다. (합성음성 문제는 0/1 라벨을 제출하므로 이 변환이 곧 점수가 됨)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

Xt = torch.from_numpy(X)[:, None]; Yt = torch.from_numpy(y).float()
tr_i, va_i = train_test_split(np.arange(len(X)), test_size=0.15, random_state=42, stratify=y)
EPOCHS = 40; BS = 64

torch.manual_seed(0)
model = AudioCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3); bce = nn.BCEWithLogitsLoss()
for epoch in range(EPOCHS):
    model.train(); perm = np.random.RandomState(epoch).permutation(tr_i)
    for i in range(0, len(perm), BS):
        b = perm[i:i+BS]
        opt.zero_grad(); loss = bce(model(Xt[b].to(device)).squeeze(1), Yt[b].to(device))
        loss.backward(); opt.step()
model.eval()
with torch.no_grad():
    p_va = torch.sigmoid(model(Xt[va_i].to(device)).squeeze(1)).cpu().numpy()

thr = max(np.linspace(0.1, 0.9, 81), key=lambda t: f1_score(y[va_i], p_va > t))
print(f"val AUC = {roc_auc_score(y[va_i], p_va):.4f}")
print(f"F1@0.5  = {f1_score(y[va_i], p_va > 0.5):.4f}   →   F1@{thr:.2f}(튜닝) = {f1_score(y[va_i], p_va > thr):.4f}"
      f"   | acc = {accuracy_score(y[va_i], p_va > thr):.4f}")
print("→ 2D 오디오 특징을 이미지처럼 CNN 으로 분류 + 임계값 튜닝 — 합성음성 문제의 멜+CNN(F1) 과 동형.")

## 6. 캐글 제출 파일 생성 (`vid_id,is_turkey` 확률)
이 대회 지표는 **AUC** 라 *확률*을 그대로 제출한다. (라벨 0/1 을 요구하는 합성음성 문제라면 위 임계값으로 이진화)

In [ ]:
Xte = torch.from_numpy(X_test)[:, None]
model.eval(); probs = []
with torch.no_grad():
    for i in range(0, len(Xte), 256):
        probs.append(torch.sigmoid(model(Xte[i:i+256].to(device)).squeeze(1)).cpu().numpy())
probs = np.concatenate(probs)

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"vid_id": test_ids, "is_turkey": probs}).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(probs))
pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [Don't Call Me Turkey!](https://www.kaggle.com/c/dont-call-me-turkey/submit) 에 제출하세요.

**기법 요약** (IOAI 2025 Synthetic Speech Detector): 오디오 분류를 **비전 문제**로 — 파형을 **2D 시간-주파수
표현**(멜 스펙트로그램; 여기선 VGGish 특징)으로 바꿔 **1채널 CNN** 으로 분류. AUC 가 높아도 F1 은 **임계값 튜닝**
이 필요(불균형). 합성음성 문제의 *미리 변환된 멜 `.pt` + ResNet(1ch conv1) + F1* 과 동형. 더 끌어올리려면:
더 깊은 CNN(ResNet18/34, `weights=None`), 프레임축 마스킹·증강(SpecAugment), 에폭/배치/학습률 튜닝, 패딩 마스크 처리.